In [1]:
# 1. Dọn dẹp sạch sẽ các phiên bản cũ
%pip uninstall -y transformers huggingface_hub peft qwen-vl-utils bitsandbytes

# 2. Cài đặt các thư viện nền tảng (ẩn log cho gọn)
%pip install -q torch torchvision torchaudio accelerate tqdm

# 3. Cài đặt thư viện lõi (BỎ -q để theo dõi xem mạng Kaggle có tải thành công không)
%pip install --upgrade transformers huggingface_hub peft qwen-vl-utils

# 4. Cài đặt CUDA và bitsandbytes
%pip install -q nvidia-nvjitlink-cu12
%pip install -q --upgrade bitsandbytes

Could not import runpy module
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap>", line 1176, in exec_module
  File "<frozen runpy>", line 15, in <module>
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap>", line 1176, in exec_module
  File "<frozen importlib.util>", line 16, in <module>
  File "C:\Users\hi\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\threading.py", line 35, in <module>
    _start_joinable_thread = _thread.start_joinable_thread
                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module '_thread' ha

In [1]:
# ==========================================
# 1. IMPORT THƯ VIỆN (Bắt buộc chạy lại sau khi Restart Runtime)
# ==========================================
import json
import os
import re
import gc
import time
import torch
from tqdm import tqdm
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
from qwen_vl_utils import process_vision_info

# ==========================================
# 2. CẤU HÌNH VÀ TẢI MODEL
# ==========================================
INPUT_PATH = "/kaggle/input/datasets/nhimchauphii/hoangh/test_part1_200.json"   
OUTPUT_PATH = "t3_qwen3_vl_8b_instruct_outputs_part1.json" 

BASE_MODEL_ID = "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit" 
LORA_MODEL_ID = "Nhat-Quang/outfitmatch-stylist-final-qwen3vl8b-instruct-lora"

print("🔄 Đang cấu hình và tải Base Model...")

# Tải Processor
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

# Cấu hình 4-bit để tải mô hình lượng tử hóa
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Tải Base Model bằng Qwen3VLForConditionalGeneration với cấu hình lượng tử hóa
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

print(f"Đang đắp LoRA adapter ({LORA_MODEL_ID}) lên Base Model...")
model = PeftModel.from_pretrained(base_model, LORA_MODEL_ID)
model.eval()
print("Tải Model thành công!\n")

ModuleNotFoundError: No module named 'qwen_vl_utils'

In [ ]:
# ==========================================
# 2. HÀM TÌM KIẾM TÀI LIỆU (RETRIEVAL)
# ==========================================
def retrieve_documents(item):
    """
    Nhiệm vụ: Lấy trực tiếp tập ngữ cảnh chuẩn đã được gán nhãn sẵn trong file test.
    """
    return item.get("contexts", [])


In [ ]:
# ==========================================
# 3. HÀM SINH CÂU TRẢ LỜI (Ô SỐ 6)
# ==========================================
def generate_answer(question: str, retrieved_contexts: list) -> str:
    if isinstance(retrieved_contexts, list):
        context_text = "\n- ".join([str(c) for c in retrieved_contexts])
    else:
        context_text = str(retrieved_contexts)

    if not context_text.strip():
        return "Tôi không tìm thấy thông tin."

    # Định dạng tin nhắn chuẩn của Qwen-VL
    messages = [
        {
            "role": "system", 
            "content": [{"type": "text", "text": "Bạn là chuyên gia tư vấn thời trang. Hãy trả lời câu hỏi CHỈ DỰA TRÊN tài liệu tham khảo. Nếu tài liệu không chứa thông tin, hãy nói chính xác: 'Tôi không tìm thấy thông tin'."}]
        },
        {
            "role": "user", 
            "content": [{"type": "text", "text": f"--- TÀI LIỆU THAM KHẢO ---\n{context_text}\n--------------------------\nCâu hỏi: {question}"}]
        }
    ]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages) # Dù không có ảnh, vẫn cần hàm này để parse chuẩn
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    # Tính toán chiều dài đầu vào để loại bỏ prompt lúc in kết quả
    input_length = inputs["input_ids"].shape[1]
    
    # Lấy ID của thẻ đóng chat <|im_end|> để ép model dừng đúng lúc
    im_end_id = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
    eos_ids = [processor.tokenizer.eos_token_id]
    if im_end_id is not None and not isinstance(im_end_id, list):
        eos_ids.append(im_end_id)
    elif isinstance(im_end_id, list):
        eos_ids.extend(im_end_id)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=512, 
            temperature=0.1,    
            do_sample=True,
            top_p=0.9,
            use_cache=True,
            eos_token_id=eos_ids
        )
        
    generated_tokens = outputs[0][input_length:]
    response = processor.decode(generated_tokens, skip_special_tokens=True)
    
    return response.strip()


In [ ]:
# ==========================================
# 4. VÒNG LẶP CHÍNH (Ô SỐ 7)
# ==========================================
def run_evaluation_pipeline(input_file: str, output_file: str, save_interval: int = 10, limit: int = None):
    dataset = []
    
    # 1. Đọc dữ liệu (Checkpoint hoặc Khởi tạo mới)
    if os.path.exists(output_file):
        print(f"🔄 Đang tải Checkpoint từ: {output_file}...")
        with open(output_file, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
    else:
        print(f"📖 Đang đọc file gốc: {input_file}...")
        with open(input_file, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
            
    if limit is not None:
        print(f"⚠️ Giới hạn xử lý {limit} câu đầu tiên để test.")
        dataset = dataset[:limit]
            
    uncompleted_items = [item for item in dataset if "answer" not in item]
    completed_samples = len(dataset) - len(uncompleted_items)
    
    print(f"🚀 Tiến trình: {completed_samples}/{len(dataset)} câu đã hoàn thành.")
    
    # 2. Chạy vòng lặp
    for idx, item in enumerate(tqdm(
        uncompleted_items, 
        desc="Đang sinh câu trả lời", 
        initial=completed_samples, 
        total=len(dataset)
    )):
        question = item.get("question", "")
        
        # Bốc context có sẵn
        retrieved = retrieve_documents(item)
        item["retrieved_contexts"] = retrieved
        
        # Gọi model
        try:
            item["answer"] = generate_answer(question, retrieved)
        except Exception as e:
            item["answer"] = f"Lỗi sinh text: {str(e)}"
        
        # 3. CHỈ LƯU CHECKPOINT SAU MỖI 10 CÂU (Bảo vệ ổ cứng Kaggle)
        if (idx + 1) % save_interval == 0 or (idx + 1) == len(uncompleted_items):
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(dataset, f, ensure_ascii=False, indent=2)
                
        # Dọn rác bộ nhớ sau mỗi câu hỏi để tránh lỗi Out Of Memory (OOM) GPU
        gc.collect()
        torch.cuda.empty_cache()
                
    print(f"\n🎉 Hoàn thành! Kết quả lưu tại: {output_file}")


In [ ]:
# Gọi hàm để bắt đầu chạy vòng lặp 200 câu
run_evaluation_pipeline(INPUT_PATH, OUTPUT_PATH, limit=5)
